# Preparing and Executing a NOAA NGEN Simulation



The purpose of this notebook is to demonstrate how CIROH-developed tools can be used to construct and execute a model simulation within the NOAA Next Generation (NGEN) modeling framework.



**Requirements**

* NGIAB Data Preprocessor



#### TODO: Complete Header 

## 0. Introduction to the NGIAB Data Preprocessor Tools

The NGIAB Data Preprocessor tools are designed to assist in the prepartion of data needed to run NextGen model simulations. The outputs of this tool are specifically designed to execute in the NextGen-in-a-Box (NGIAB) system.

It uses geometry and model attributes from the v2.2 hydrofabric, raw forcing data can be collected from either the NWM Retrospective v3 or the AORC 1km gridded data. This tool is able to perform the following tasks:

1. Subset (delineate) the NextGen HydroFabric based on a user-provided point of interest (catchment, gage, flowpath etc).
2. Collect and prepare model forcings using the `exact extract` methodology.
3. Create the simulation configuration files needed to run a model simulation.
4. Execute model simulations using the NGIAB tools.


The `NGIAB Data Preprocessor` tools can be executed from the commandline using the sytax:

```
python -m <module name> <args>
```

for example

```
python -m ngiab_data_cli --help
```

While the `ngiab_data_cli` commands listed in this notebook can be executed within their respective cells, it is recommended that they are instead executed in a terminal window. The recommended way to do this is by opening a Jupyter Terminal window and docking it alongside the notebook instructions, for example.

![Jupyter Screenshot](./img/jupyter-screenshot.png)


## 1. Collect HydroFabric Data

Open the USGS map view to identify a location of interest. Note the stream gage number because this will be used to collect HydroFabric data for our NGEN simulation. [USGS National Water Information System Map](https://maps.waterdata.usgs.gov/mapper/index.html)


![USGS Gage Location](./img/usgs-gage-location.png)

In [ ]:
# define our site number for later.
usgs_site_number = '02464000'

Subset the HydroFabric at our desired gage using the following command:

```
python -m ngiab_data_cli -i gage-02464000 --subset
```

|Command| Description|
|---|---|
|`-m ngiab_data_cli` | tells Python which module it should run   |
|`-i gage-02464000`  | the input feature to subset   |
|`--subset`          | indicates that the hydrofabric subsetting operation should be  performed. |

---

The same command can be executed within the notebook using the following command:

<div style="border-left: 3px solid red; padding-left: 10px;">
<pre>
usgs_site_number = '02464000'
<br>
!python -m ngiab_data_cli -i gage-{usgs_site_number} --subset

</pre>
</div>

We can preview these data using `Fiona` and `GeoPandas`.

In [ ]:
import fiona
import geopandas as gpd
import matplotlib.pyplot as plt

# define the path to our geopackage file
hf_filepath = f"./gage-{usgs_site_number}/config/gage-{usgs_site_number}_subset.gpkg"

Since our geopackage may contain more than one layer, we'll first use `fiona` to list the layers that are available.

In [ ]:
layers = fiona.listlayers(hf_filepath)

print( 45*'-' + '\nThe Geopackage contains the following layers: \n' + 45*'-')
for layer in layers:
    print(layer)

Let's load and plot the `divides`, `flowpaths`, and `nexus` layers.

In [ ]:
# load the layers
divides = gpd.read_file(hf_filepath, layer="divides")
flowpaths = gpd.read_file(hf_filepath, layer="flowpaths")
nexus = gpd.read_file(hf_filepath, layer="nexus")

print(f'{len(divides)} Divides')
print(f'{len(flowpaths)} Flowpaths')
print(f'{len(nexus)} Nexus')

In [ ]:
# create a figure
fig, ax = plt.subplots(figsize=(10, 10))

divides.plot(ax=ax, color='green', alpha=0.3, edgecolor='darkgreen')
flowpaths.plot(ax=ax, color='blue', edgecolor='blue', alpha=0.5, label='HydroFabric Reaches')
nexus.plot(ax=ax, color='red', edgecolor='red', alpha=0.5, label='HydroFabric Nexus')

plt.title(f'NGEN HydroFabric at USGS:{usgs_site_number}')
ax.legend()
plt.show()

Each of these GeoPandas dataframes contains attributes that we can look at as well.

In [ ]:
divides

We can add attach additional attributes to these divide features.

In [ ]:
divide_attrs = gpd.read_file(hf_filepath, layer="divide-attributes")
merged_divides = divides.merge(divide_attrs, on='divide_id')

merged_divides

Notice there are 50 columns as opposed to the 11 in the original dataframe. We list all of the columns too.

In [ ]:
for c in merged_divides.columns:
    print(c)

We can do the same thing for the flowpaths.

In [ ]:
flow_attrs = gpd.read_file(hf_filepath, layer="flowpath-attributes")
merged_flows = flowpaths.merge(flow_attrs, on='id')
for c in merged_flows.columns:
    print(c)

## 2. Collect Model Forcings

Now that we have an NGEN HydroFabric for our location of interest, we can collect meteorlogical forcing data. This can be done using the `NGIAB Data Preprocessor`.

```
python -m ngiab_data_cli -i gage-02464000 -f --start 2022-01-01 --end 2022-12-31
```


|Command| Description|
|---|---|
|`-m ngiab_data_cli` | tells Python which module it should run   |
|`-i gage-02464000`  | the input feature to subset   |
|`-f`          | indicates that the forcing collection operation should be executed |
|`--start 2022-01-01`          | the start time for data collection |
|`--end 2022-12-31`          | the end time for data collection |

---

The same command can be executed within the notebook using the following command:

<div style="border-left: 3px solid red; padding-left: 10px;">
<pre>
usgs_site_number = '02464000'
<br>
!python -m ngiab_data_cli -i gage-{usgs_site_number} -f --start 2022-01-01 --end 2022-12-31

</pre>
</div> 

We can preview these data using the `xarray` library.

In [ ]:
import xarray

In [ ]:
forcing_path = f'./gage-{usgs_site_number}/forcings/forcings.nc'

In [ ]:
ds = xarray.open_dataset(forcing_path)
ds

**Summary Forcing Variables**

|Variable Name |	Description	|Units |
|---|---|---|
|APCP_surface	|Precipitation at the surface level	Meters (m)|
|DLWRF_surface	|Downward Longwave Radiation Flux at the surface	|Watts per square meter (W/m²)|
|PRES_surface	|Air Pressure at the surface|	Pascals (Pa)|
|SPFH_2maboveground	|Specific Humidity at 2 meters Above Ground Level	| Kilograms per kilogram (kg/kg)|
|precip_rate	|Precipitation rate at the surface	|Meters per second (m/s)|
|DSWRF_surface|	Downward Shortwave Radiation Flux at the surface	|Watts per square meter (W/m²)|
|TMP_2maboveground	|Temperature at 2 meters Above Ground Level|	Kelvin (K)|
|UGRD_10maboveground	|Eastward (U) Wind component at 10 meters Above Ground Level|	Meters per second (m/s)|
|VGRD_10maboveground|	Northward (V) Wind component at 10 meters Above Ground Level	|Meters per second (m/s)|

We can visualize these data at any of the catchments within our domain. This is done by isolating our area of interest using xarray indexing, and then leveraging matplotlib to construct the plot. 

We'll need to provide a `catchment-id` for the data we want to plot. All of the catchment ids can be printed using the following command:

```
ds.ids.values
```

Alternatively, we can copy the catchment id associated with the outlet of our domain from the `ngiab_data_cli` output in the terminal, i.e. `cat-485431`

In [ ]:
outlet_catchment_id = 'cat-485431'

# create a coordinate containing the values of `ids`
ds = ds.assign_coords({'catchment-id': ds.ids})

In [ ]:
ds_cat = ds.sel({"catchment-id": outlet_catchment_id})
ds_cat

In [ ]:
top_variable = 'precip_rate'
bottom_variable = 'TMP_2maboveground'

fig, ax1 = plt.subplots(figsize=(10, 5))

ds_cat[bottom_variable].plot(ax=ax1, label=bottom_variable, color='grey')
bottom_variable_units = ds_cat[bottom_variable].attrs['units']
ax1.set_ylabel(f'{bottom_variable} ({bottom_variable_units})')
ax1.tick_params(axis='y')
ax1.set_ylim((ds_cat[bottom_variable].min().item(),
              ds_cat[bottom_variable].max().item() * 1.1)) 

# Create a second y-axis
ax2 = ax1.twinx()

# Second series on right y-axis
ds_cat[top_variable].plot(ax=ax2, label=top_variable, color='blue')
top_variable_units = ds_cat[top_variable].attrs['units']
ax2.set_ylabel(f'{top_variable} ({top_variable_units})')
ax2.tick_params(axis='y')
ax2.set_ylim((ds_cat[top_variable].min(),
              ds_cat[top_variable].max().item() * 4)) 
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 3. Generate Model Configuration Files

Now that we have an NGEN HydroFabric for our location of interest and the forcing data for our time period, we're ready to configure the model using the `NGIAB Data Preprocessor`.

```
python -m ngiab_data_cli -i gage-02464000 -r --start 2022-01-01 --end 2022-12-31
```


|Command| Description|
|---|---|
|`-m ngiab_data_cli` | tells Python which module it should run   |
|`-i gage-02464000`  | the input feature to subset   |
|`-r`          | indicates that the configuration files should be created |
|`--start 2022-01-01`          | the start time for our simulation|
|`--end 2022-12-31`          | the end time for our simulation|

---

The same command can be executed within the notebook using the following command:

<div style="border-left: 3px solid red; padding-left: 10px;">
<pre>
usgs_site_number = '02464000'
<br>
!python -m ngiab_data_cli -i gage-{usgs_site_number} -r --start 2022-01-01 --end 2022-12-31

</pre>
</div> 

List the configuration files that were generate by this previous command:

```
tree "$(pwd)/gage-02464000/config"
```


The same command can be executed within the notebook using the following command:
<div style="border-left: 3px solid red; padding-left: 10px;">
<pre>
tree $(pwd)/gage-{usgs_site_number}/config
</pre>
</div> 

#### TODO: Explain Model Configuration Files.

## 4. Executing a Simulation

Move into the directory where our model simulation data exists.

```
cd ./gage-02464000
```

Run the simulation using the `ngen` command. This command requires several inputs:

`ngen <catchment_data_path> <catchment subset ids> <nexus_data_path> <nexus subset ids> <realization_config_path>`

where: 

- catchment_data_path - the path to the catchment input data 
- catchment subset ids - defines the catchments to be included in the simulation (a list of comma seperated ids or "" to run all catchments)
- nexus_data_path - the path to the nexus input data
- nexus subset ids - defines the nexus to be included in the simulation (a list of comma seperated ids or "" to run all nexus)
- realization_config_path - path to the realization configuration file for the simulation.

```
ngen ./config/gage-02464000_subset.gpkg "" \
     ./config/gage-02464000_subset.gpkg "" \
     ./config/realization.json
```

The same command can be executed within the notebook using the following command:

<div style="border-left: 3px solid red; padding-left: 10px;">
<pre>
usgs_site_number = '02464000'
<br>
!ngen ./config/gage-{usgs_site_number}_subset.gpkg "" \
      ./config/gage-{usgs_site_number}_subset.gpkg "" \
      ./config/realization.json

</pre>
</div> 

## 5. Evaluating Simulation Results

Our NextGen simulation has created a number of different outputs. These outputs are categorized as either (1) NextGen model outputs or () T-route model outputs; all of them can be found in the `outputs` directory. Below is a high-level summary of the output variables that our simulation produced.


**NextGen Model Outputs**

|Variable Name	|Description	|Units|
|---|---|---|
|RAIN_RATE|	Rainfall Rate	|Meters (m)|
|GIUH_RUNOFF	|Lagged and attenuated runoff using the Geomorphological Instantaneous Unit Hydrograph	|Meters (m)|
|INFILTRATION_EXCESS	|Amount of rainfall that exceeds infiltration capacity	|Meters (m)|
|DIRECT_RUNOFF|	Surface runoff from rainfall/throughfall input|	Meters (m)|
|NASH_LATERAL_RUNOFF|	Lateral subsurface flow through the Nash cascade	|Meters (m)|
|DEEP_GW_TO_CHANNEL_FLUX|	Flux from deep groundwater to the channel	|Meters (m)|
|SOIL_TO_GW_FLUX	|Flux from soil to groundwater|	Meters (m)|
|Q_OUT	|Outflow from the catchment	|Meters (m)|
|POTENTIAL_ET|	Potential evapotranspiration|	Meters (m)|
|ACTUAL_ET	|Actual evapotranspiration	|Meters (m)|
|GW_STORAGE	|Groundwater storage	|Meters (m)|
|SOIL_STORAGE	|Soil moisture storage	|Meters (m)|
|SOIL_STORAGE_CHANGE	|Change in soil moisture storage	|Meters (m)|
|SURF_RUNOFF_SCHEME	|Surface runoff as determined by the runoff scheme	|unitless|
|NWM_PONDED_DEPTH	|Ponded depth of water based on NWM model	|Meters (m)|

*Note: this metadata was collected by David Tarboton in 2025*

---

**T-Route Model Outputs**

|Variable Name	|Description	|Units|
|---|---|---|
|feature_id| Reach segment identifier | unitless |
|flow| River streamflow | Cubic Meters per Second (m3 s-1) |
|velocity| River velocity | Meters per Second (m/s)|
|depth| River depth| Meteres (m) |
|nudge| River streamflow nudge values | Cubic Meters per Second (m3 s-1) |

Load the t-route output file into memory using `xarray`

In [ ]:
t_route_output = f'./gage-{usgs_site_number}/outputs/troute/troute_output_202201010000.nc'

In [ ]:
ds = xarray.open_dataset(t_route_output)
ds

We'll need to use a `feature_id` to query and plot simulation results from the t-route output file. One way that we can accomplish this through the relationships defined in the HydroFabric. Specifically, the `divides` and `flowpaths` both contain identifiers that can be used to derive the `feature_id` value.

In [ ]:
flowpaths

The `catchment_id` can be used to query the t-route outputs in the Xarray Dataset that we loaded above. The only difference between these fields is that the former contains a prefix of `cat-` which when removed maps directly to the `feature_id` variable in the t-route output. For example, a catchment value of `cat-485431` maps to a feature_id of `485431`.

In [ ]:
outlet_catchment_id = 'cat-485431'
outlet_feature_id = int(outlet_catchment_id.split('-')[-1])

print(f'The feature_id of the domain outlet = {outlet_feature_id}')

Now we can isolate the variables associated with this reach and create a plot.

In [ ]:
dat = ds.sel(feature_id = outlet_feature_id)

dat.flow.plot();

### 6. Where to Go Next?

- NGIAB Data PreProcessor
- NGIAB
- DataStream
- NGIAB-cal
